# TP2 : Recherche hybride et reranking

**Contexte :** assistant du service client de Lagune Banque.

La recherche vectorielle retrouve le sens, mais elle classe parfois le bon chunk trop bas et peut manquer un terme exact. Vous allez la combiner avec une recherche par mots-clés (BM25), ajouter un reranker qui reclasse les meilleurs candidats, puis mesurer ce que chaque technique apporte.

```mermaid
flowchart LR
    Q["Question"] --> V["Recherche vectorielle"]
    Q --> B["Recherche BM25"]
    V --> F["Fusion RRF"]
    B --> F
    F --> RR["Reranker"]
    RR --> S["Meilleurs chunks"]
```

Complétez les `# TODO`.

## <a id="syntaxe"></a>Syntaxe

**Recherche vectorielle.** `retrieve(question, n)`, définie plus bas, renvoie les `n` chunks les plus proches de la question, du plus proche au plus lointain : `[{"texte": ..., "distance": ..., "meta": {...}}, ...]`.

**BM25.** `ra.BM25(chunks)` indexe les chunks une fois ; `.search(question, n)` renvoie les `n` chunks qui partagent le plus de mots rares avec la question, dans le même format.

**Reranker.** `litellm.rerank` envoie la question et les textes à reclasser au modèle de reranking. Le résultat `r.results` est trié du plus pertinent au moins pertinent ; chaque élément donne la position du texte dans la liste envoyée (`index`) et son score (`relevance_score`) :

```python
r = litellm.rerank(model="...", query="Quel est le taux ?", documents=["texte A", "texte B", "texte C"], top_n=2, api_key="...")
r.results   # [{"index": 2, "relevance_score": 0.91}, {"index": 0, "relevance_score": 0.35}]
```

`index` renvoie à la liste `documents` : c'est lui qui permet de retrouver le chunk d'origine.

## <a id="configuration"></a>Configuration

In [1]:
import os
import sys
from pathlib import Path

import litellm
import pandas as pd

sys.path.insert(0, str(next(p for p in [Path("code_partage"), Path("../code_partage"), Path("../../code_partage")] if (p / "rag.py").exists()).resolve()))
import rag
import rag_advanced as ra
from config import prepare
from weaviate_store import connect

In [2]:
prepare(embedding=True, rerank=True)

Connexion OK : gemini/gemini-3.5-flash-lite gemini/gemini-embedding-2 azure_ai/Cohere-rerank-v4.0-fast


## <a id="index-questions"></a>Index et questions

L'index est celui retenu au TP1 : splitter récursif, 500 caractères, overlap 75. Les questions viennent de `data/evaluation/questions_rag.json` : 12 cas courants et 35 cas difficiles.

Le reranker est un service payant à débit limité, partagé par toute la salle. La mesure se fait donc sur un **échantillon de 15 questions** : 6 cas courants et 9 cas difficiles. Sur 15 questions, un écart de MRR inférieur à 0,1 n'est pas significatif.

In [3]:
documents = rag.load_corpus()
for d in documents:
    if d["fichier"].endswith(".pdf") and not d["avertissement"]:
        d["pages"] = rag.strip_headers(d["pages"])
chunks = rag.split_documents(documents, 500, 75, "hybride")
index = rag.index_chunks(connect(), "index_hybride", chunks)

courants = ra.load_questions("representatif")[::2]
difficiles = ra.load_questions("difficile")[::4]
echantillon = courants + difficiles
len(echantillon)

15

## <a id="recherche-vectorielle"></a>Recherche vectorielle

`retrieve` renvoie les `n` chunks les plus proches. `ra.measure` applique une recherche à chaque question et calcule le **MRR** : la moyenne de `1 / rang`, où le rang est la position du premier chunk qui contient la réponse complète. 1,00 signifie que la bonne réponse est toujours en première position.

In [4]:
def retrieve(question, n=10):
    return rag.search(index, question, n, rag.AUJOURDHUI)


base = ra.measure(lambda q, h: retrieve(q), echantillon)
{cle: base[cle] for cle in ("MRR", "en 1er", "dans les 3 premiers", "absents")}

{'MRR': 0.65, 'en 1er': 6, 'dans les 3 premiers': 14, 'absents': 0}

## <a id="bm25"></a>Recherche par mots-clés : BM25

BM25 note un chunk d'autant plus qu'il contient les mots de la question, en donnant plus de poids aux mots rares. Cette méthode retrouve les termes exacts (un numéro de téléphone, un code), mais ne comprend pas les synonymes. `ra.BM25` l'implémente.

Une question dont les mots rares (« rémunération », « Épargne Plus ») figurent dans le bon chunk : la recherche par mots-clés le place en tête.

In [5]:
bm25 = ra.BM25(chunks)


def compare(question, elements):
    print(question)
    for nom, resultats in (("vectoriel", retrieve(question, 3)), ("BM25", bm25.search(question, 3))):
        print(f"  {nom}")
        for rang, p in enumerate(resultats, 1):
            reponse = "<- contient la réponse" if ra.rank([p], elements) else ""
            print(f"    [{rang}] {p['meta']['source']} | {p['texte'][:50]!r} {reponse}")


compare("Quel est le taux de rémunération de l'Épargne Plus ?", ["3,5 %"])

Quel est le taux de rémunération de l'Épargne Plus ?
  vectoriel
    [1] DOC-04 | 'Fiche produit – Épargne Plus\n\nIdentifiant : DOC-04' 
    [2] DOC-06 | "Note 6 – Après 6 prêts remboursés à l'échéance, le" 
    [3] DOC-05 | 'Quand les intérêts sont-ils versés ?\n\nLes intérêts' 
  BM25
    [1] DOC-05 | 'Quand puis-je redemander un prêt après un impayé ?' <- contient la réponse
    [2] DOC-04 | 'Fiche produit – Épargne Plus\n\nIdentifiant : DOC-04' 
    [3] DOC-05 | 'Quand les intérêts sont-ils versés ?\n\nLes intérêts' 


Le chunk retrouvé par BM25 commence par une autre question de la FAQ : le taux de 3,5 % figure plus loin, dans le même chunk de 500 caractères (c'est l'effet du chunking vu au TP1). La recherche vectorielle classe en tête la fiche produit, qui ne contient pas le taux.

Une question en anglais : elle n'a aucun mot en commun avec les chunks, et c'est maintenant la recherche par mots-clés qui échoue.

In [6]:
compare("How often is interest paid on the savings account?", ["trimestriellement"])

How often is interest paid on the savings account?
  vectoriel
    [1] DOC-04 | 'Rémunération\n\nVotre épargne est rémunérée au taux ' <- contient la réponse
    [2] DOC-06 | 'Élément\nValeur\nDépôt minimum\n1 000 FCFA\nSolde maxi' 
    [3] DOC-05 | 'Quand les intérêts sont-ils versés ?\n\nLes intérêts' 
  BM25
    [1] DOC-08 | '11. Situations particulières\nPlusieurs incidents l' 
    [2] DOC-07 | '11. Situations particulières\nPlusieurs incidents l' 


Chaque méthode comble les angles morts de l'autre. Reste à les combiner.

## <a id="fusion-rrf"></a>Fusion des classements : RRF

On ne peut pas additionner une distance vectorielle et un score BM25 : leurs échelles n'ont rien en commun. La fusion **RRF** (Reciprocal Rank Fusion) n'utilise que les rangs : un chunk reçoit `1 / (k + rang)` dans chaque liste, avec `k = 60`, et on additionne. Un chunk bien classé dans les deux listes remonte.

**TODO 1** : dans `rrf(listes, k=60, n=10)`, complétez la ligne qui ajoute la contribution de chaque rang au score du chunk. `listes` est une liste de listes de chunks, chacune triée du meilleur au moins bon ; un chunk est identifié par son texte. La fonction renvoie les `n` chunks de plus grand score.

In [18]:
def rrf(listes, k=60, n=10):
    scores, chunks = {}, {}
    for liste in listes:
        for rang, p in enumerate(liste, 1):    # rang 1 = le meilleur
            chunks[p["texte"]] = p
            scores[p["texte"]] = scores.get(p["texte"], 0) + 1 / (k + rang)
    meilleurs = sorted(scores, key=scores.get, reverse=True)[:n]
    return [chunks[t] for t in meilleurs]


Test sur trois chunks et deux listes. Attendu : `['a', 'c', 'b']`.

In [19]:
a, b, c = {"texte": "a"}, {"texte": "b"}, {"texte": "c"}
[p["texte"] for p in rrf([[a, b, c], [c, a, b]])]

['a', 'c', 'b']

## <a id="recherche-hybride"></a>Recherche hybride

**TODO 2** : écrivez `hybrid(question)` : la fusion RRF de la recherche vectorielle (`retrieve`) et de la recherche par mots-clés (`bm25.search`).

In [20]:
from weaviate import client


def hybrid(question):
    vector_chunks = retrieve(question)
    bm25_chunks = bm25.search(question)
    return rrf([vector_chunks, bm25_chunks])


## <a id="reranking"></a>Reranking

Un **reranker** est un modèle qui lit la question et un chunk ensemble, et note leur pertinence. Il est plus précis qu'un embedding, qui encode la question et le chunk séparément, mais il est trop lent pour parcourir toute la base. On procède donc en deux étapes : la recherche vectorielle ramène 20 candidats, le reranker les reclasse, et on garde les 5 meilleurs.

Le modèle, la clé et l'adresse du reranker viennent de l'environnement : `MODELE_RERANK`, `RERANK_API_KEY` et, s'il est défini, `RERANK_API_BASE`, placé dans `parametres` pour être passé à `api_base`.

**TODO 3** : dans `rerank_chunks(question, chunks, n=5)`, complétez l'appel à `litellm.rerank` : la question, les textes des chunks (`textes`), `top_n=n`, la clé et `**parametres`. La suite est fournie : elle renvoie les `n` chunks les mieux classés, du meilleur au moins bon, chacun avec une clé `score` (le `relevance_score`), en retrouvant chaque chunk grâce à `index`. Cette version ne gère pas les relances en cas de limite de débit : `rag.rerank` le fait, pas votre fonction.

In [21]:
def rerank_chunks(question, chunks, n=5):
    textes = [p["texte"] for p in chunks]
    parametres = {"api_base": os.environ["RERANK_API_BASE"]} if os.getenv("RERANK_API_BASE") else {}
    r = litellm.rerank(model=os.environ["MODELE_RERANK"], query=question, documents=textes, top_n=n,
                       api_key=os.environ["RERANK_API_KEY"], **parametres)
    # chaque résultat désigne un chunk par sa position dans `chunks` (index) : on renvoie ce chunk avec son score
    return [{**chunks[x["index"]], "score": x["relevance_score"]} for x in r.results]


**TODO 4** : écrivez `two_stage(question)` : les 20 premiers chunks de la recherche vectorielle, reclassés par `rerank_chunks`, dont on garde les 5 meilleurs.

In [22]:
def two_stage(question):
    candidats = retrieve(question, 20)
    return rerank_chunks(question, candidats, 5)


Exemple : le reranker donne à chaque chunk un score de pertinence.

In [23]:
for p in two_stage("Combien de temps avant de pouvoir réemprunter ?")[:3]:
    print(f"{p['score']:.2f} | {p['meta']['source']} | {p['texte'][:90]!r}")

0.84 | DOC-02 | 'Quand puis-je emprunter à nouveau ?\n\nVous pouvez emprunter à nouveau 48 heures après le re'
0.76 | DOC-05 | 'Quand puis-je redemander un prêt après un impayé ?\n\nAprès le remboursement intégral, vous '
0.74 | DOC-05 | "Puis-je avoir deux prêts en cours en même temps ?\n\nNon. Vous ne pouvez avoir qu'un seul Pr"


## <a id="mesures"></a>Mesures

Chaque configuration est appliquée aux 15 questions de l'échantillon. La dernière combine la recherche hybride et le reranker : 20 candidats issus de la fusion, reclassés.

In [24]:
CONFIGURATIONS = {
    "vectoriel": lambda q, h: retrieve(q),
    "BM25": lambda q, h: bm25.search(q),
    "hybride": lambda q, h: hybrid(q),
    "vectoriel + reranker": lambda q, h: two_stage(q),
    "hybride + reranker": lambda q, h: rerank_chunks(q, rrf([retrieve(q, 20), bm25.search(q, 20)], n=20), 5),
}
resultats = {nom: ra.measure(recherche, echantillon) for nom, recherche in CONFIGURATIONS.items()}


def mrr(rangs):
    return round(sum(1 / r for r in rangs if r) / len(rangs), 2)


pd.DataFrame({
    nom: {"MRR": r["MRR"], "en 1er": f"{r['en 1er']} / {len(echantillon)}", "absents": r["absents"],
          "MRR cas courants": mrr(r["rangs"][:len(courants)]), "MRR cas difficiles": mrr(r["rangs"][len(courants):]),
          "secondes par question": r["secondes"]} for nom, r in resultats.items()}).T

,MRR,en 1er,absents,MRR cas courants,MRR cas difficiles,secondes par question
vectoriel,0.65,6 / 15,0,0.81,0.54,0.66
BM25,0.66,8 / 15,2,0.65,0.67,0.0
hybride,0.7,9 / 15,0,0.78,0.65,0.65
vectoriel + reranker,0.79,10 / 15,1,0.83,0.76,2.24
hybride + reranker,0.78,10 / 15,1,0.83,0.74,2.37


## Contrôle automatique

La dernière cellule affiche ÉCHEC tant que les `# TODO` ne sont pas faits.

In [25]:
try:
    assert [p["texte"] for p in rrf([[a, b, c], [c, a, b]])] == ["a", "c", "b"], "rrf : voir l'exemple avec trois chunks (TODO 1)"
    assert hybrid("Quel est le plafond initial du Prêt Express ?") and len(hybrid("Quel est le plafond initial du Prêt Express ?")) <= 10, "hybride doit renvoyer au plus 10 chunks (TODO 2)"
    question = "Quel est le plafond initial du Prêt Express ?"
    candidats = retrieve(question, 20)
    classes = rerank_chunks(question, candidats, 5)
    assert len(classes) == 5 and all("score" in p for p in classes), "rerank_chunks : 5 chunks, chacun avec son score (TODO 3)"
    assert [p["score"] for p in classes] == sorted((p["score"] for p in classes), reverse=True), "rerank_chunks : du meilleur au moins bon (TODO 3)"
    assert all(p["texte"] in {x["texte"] for x in candidats} for p in classes), "rerank_chunks : renvoyez les chunks d'origine (TODO 3)"
    candidats = two_stage(question)
    assert len(candidats) == 5 and all("score" in p for p in candidats), "two_stage : 5 chunks, chacun avec son score (TODO 4)"
    print("OK")
except Exception as e:
    print(f"ÉCHEC : {e if isinstance(e, AssertionError) else 'terminez les # TODO et relancez les cellules.'}")

OK
